# 🎬 Phase 4: Cross-Episode Synthesis Engine
### Enhanced with Semantic Embeddings, Diversity-Aware Ranking, Controlled Retries & Precise Trims
**Series:** *I Married a Broken Soldier. He was a Duke* (50 Episodes)  
**Technologies Used:** Python Case-Insensitive Search, `google-genai` SDK (v3.0+), Pydantic v2, NumPy Cosine Similarity, Pandas  

---

### Key Architectural Tenets & High-Priority Improvements:
1. **Semantic Retrieval via Embeddings (Improvement 1.1)**:
   - Moves beyond simple substring keyword matching to resolve semantic equivalents (e.g. *"soldier examines royal signet"* in Ep 7 $\leftrightarrow$ *"Duke's ancestral seal presented before court"* in Ep 42).
   - Generates vector embeddings for narrative moments and computes pairwise cosine similarity:
     $$\operatorname{sim}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$
   - Retrieves top 3–5 cross-episode moments and attaches verbatim source screenplay elements before calling the LLM.
2. **Diversity-Aware Final Ranking & Selection (Improvement 1.2)**:
   - Eliminates concentration bias (e.g., 10 recommendations from Episode 42) using a greedy diversity selection algorithm.
   - Enforces configurable constraints:
     - **Episode Concentration:** Maximum 3 recommendations per episode.
     - **Archetype Diversity:** Guaranteed representation of all 3 archetypes (`immediate_shock_opener`, `visual_anomaly_mystery_escalation`, `emotional_reversal_turning_point`).
     - **Composition Diversity:** Balanced mix of single-episode and multi-episode promos.
     - **Quality Gate:** Never selects a low-quality clip just to satisfy diversity.
3. **Structured Output Validation & Controlled Retries (Improvement 1.3)**:
   - Uses strict Pydantic schemas for request/response validation.
   - Implements automated error capture: on schema mismatch, captures errors and executes 1 targeted correction retry before falling back to safe deterministic synthesis.
4. **Precise Trim Recommendations & Transitional Editing Instructions (Improvement 1.4)**:
   - Actionable video editing blueprints for every segment: verbatim opening dialogue, ending dialogue, internal trim quotes, boundary type (`element_boundary` vs `internal_cut`), and manual waveform verification flags.
   - Transitional stitching instructions between segments. Strictly no character-count timestamp interpolation.
5. **Strict Mechanical Invariants**:
   - All segment element IDs exist in the canonical AST (`01_structured_episodes.json`).
   - Monotonic timestamps ($0 \le \text{start\_cs} < \text{end\_cs}$).
   - Hook timing: $0.0 \le \text{hook\_timestamp\_offset\_seconds} \le 3.0$.
   - Combined duration: $20.00\text{s} \le \text{total\_duration} \le 60.00\text{s}$ ($2,000 \le \text{total\_duration\_cs} \le 6,000$).
   - Multi-episode composition: Segments span $\ge 2$ distinct episodes.

```mermaid
flowchart TD
    A["01_structured_episodes.json (AST)"] --> B["02_batch_extractions.json (Moments & Context)"]
    B --> C["1.1 Semantic Embeddings & Cosine Sim"]
    C --> D["Top 3-5 Related Cross-Episode Moments"]
    A --> E["Verbatim AST Element Attachment"]
    D --> E
    E --> F["Synthesis Prompt Construction"]
    F --> G["1.3 Gemini LLM Call + Controlled Retries"]
    G --> H["Pydantic Multi-Episode Candidates"]
    H --> I["Mechanical Verification Gate (Duration, Hook, AST)"]
    I --> J["1.2 Diversity-Aware Greedy Selection (Top 15)"]
    J --> K["03_synthesized_candidates.json"]
```


## 1. Pydantic Structured Output Schemas
Defines strict data models for segment-level trims, rubric ratings, multi-episode candidates, and structured LLM responses.


In [2]:
import os
import sys
import re
import json
import time
from pathlib import Path
from typing import List, Optional, Literal, Dict, Any, Tuple, Union
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field, ValidationError

# Configure environment & directory
WORKSPACE_DIR = Path.cwd().parent if Path.cwd().name == "phase4_synthesis" else Path.cwd()
PHASE4_DIR = WORKSPACE_DIR / "phase4_synthesis"
PHASE4_DIR.mkdir(parents=True, exist_ok=True)

if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))
if str(PHASE4_DIR) not in sys.path:
    sys.path.insert(0, str(PHASE4_DIR))

# ==============================================================================
# PYDANTIC DATA MODELS
# ==============================================================================

class TrimRecommendation(BaseModel):
    """
    Precise trim recommendation for video editors (Improvement 1.4).
    Provides actionable cut instructions without character-count timestamp interpolation.
    """
    exact_opening_dialogue: str = Field(
        description="Verbatim dialogue line that begins this clip/segment (or initial action description)"
    )
    exact_ending_dialogue: str = Field(
        description="Verbatim dialogue line that ends this clip/segment before the cut"
    )
    verbatim_trim_quote: Optional[str] = Field(
        default=None,
        description="Verbatim sentence or phrase where an internal cut should be made to cut dead air"
    )
    boundary_type: Literal["element_boundary", "internal_cut"] = Field(
        description="Whether the cut aligns with an element boundary or requires an internal sentence cut"
    )
    requires_manual_editor_verification: bool = Field(
        description="True if an internal cut requires video editor verification against actual audio waveforms"
    )


class RubricScores(BaseModel):
    """Standardized 4-Criterion Rubric (1-5 integer ratings)."""
    standalone_coherence: int = Field(ge=1, le=5, description="1-5 rating: conflict/stakes clear to cold first-time viewer")
    hook_strength: int = Field(ge=1, le=5, description="1-5 rating: immediate visceral shock or intrigue within first 3 seconds")
    cliffhanger_strength: int = Field(ge=1, le=5, description="1-5 rating: cuts at peak unresolved jeopardy or open question")
    emotional_impact: int = Field(ge=1, le=5, description="1-5 rating: high-intensity betrayal, moral outrage, or dramatic reversal")


class StitchedSegment(BaseModel):
    """
    A single constituent screenplay segment within a multi-episode promo.
    Each segment is grounded in exact AST element IDs with its own trim instructions.
    """
    segment_index: int = Field(ge=1, le=3, description="1-based index (1, 2, or 3) of segment in promo sequence")
    episode_num: int = Field(description="Screenplay episode number")
    start_element_id: str = Field(description="First element ID of segment, e.g. EP01_EL010")
    end_element_id: str = Field(description="Last element ID of segment, e.g. EP01_EL019")
    start_raw: str = Field(description="Start timestamp MM:SS.ss")
    end_raw: str = Field(description="End timestamp MM:SS.ss")
    duration_seconds: float = Field(description="Duration of this segment in seconds")
    duration_cs: int = Field(description="Duration in integer centiseconds")
    purpose_in_promo: str = Field(
        description="Dramatic function of this segment (e.g. 'Hook / Initial Humiliation', 'Mystery Escalation', 'Climax / Payoff')"
    )
    trim_recommendation: TrimRecommendation = Field(
        description="Segment-specific precise trim recommendations for video editors"
    )


class MultiEpisodeCandidate(BaseModel):
    """
    A multi-episode promo candidate stitched across 2-3 episodes (composition_mode: 'multi_episode').
    Combines disparate narrative beats into a coherent 20.00s-60.00s promotional arc.
    """
    candidate_id: str = Field(description="Unique candidate identifier, e.g. SYNTH_MULTI_01")
    composition_mode: Literal["multi_episode"] = Field(
        default="multi_episode",
        description="Composition capability tag (multi_episode vs single_episode)"
    )
    archetype: Literal[
        "immediate_shock_opener",
        "visual_anomaly_mystery_escalation",
        "emotional_reversal_turning_point"
    ] = Field(description="One of the three standardized promo archetypes")
    relationship_type: Literal["explicit", "implied", "thematic"] = Field(
        description="Causal narrative connection: explicit (cause-effect), implied (secret clue/identity), or thematic (contrast baseline vs vindication)"
    )
    segments: List[StitchedSegment] = Field(
        min_length=2, max_length=3,
        description="2 to 3 constituent segments across different episodes"
    )
    transition_instructions: List[str] = Field(
        description="Precise editing transition notes between segments (e.g. 'Hard cut on Gerald dropping hand -> Flash forward audio riser -> Jump to Ep 42 cathedral smoke')"
    )
    hook_timestamp_offset_seconds: float = Field(
        ge=0.0, le=3.0,
        description="Offset within <= 3.0s from start of segment 1 where the visceral hook initiates"
    )
    hook_description: str = Field(description="Why the first 3 seconds of segment 1 stop the scrolling viewer")
    cliffhanger_description: str = Field(description="Unresolved tension or dramatic climax where segment 2/3 cuts")
    standalone_coherence_explanation: str = Field(description="Why a cold viewer understands this arc without watching intervening episodes")
    rubric_scores: RubricScores
    overall_score: int = Field(ge=4, le=20, description="Sum of the 4 rubric scores (4-20)")
    total_duration_seconds: float = Field(description="Combined duration of all segments in seconds (20.00s - 60.00s)")
    total_duration_cs: int = Field(description="Combined duration in integer centiseconds (2,000 - 6,000)")
    retrieval_method: str = Field(
        default="semantic_embedding",
        description="Method used to discover the connection: 'semantic_embedding', 'keyword_substring', or 'hybrid'"
    )
    semantic_similarity_score: Optional[float] = Field(
        default=None,
        description="Cosine similarity score (sim(A, B)) between the connected narrative moments"
    )


class SynthesisLLMResponse(BaseModel):
    """Structured response container for the synthesis LLM call."""
    candidates: List[MultiEpisodeCandidate] = Field(
        default_factory=list,
        description="List of valid synthesized multi-episode candidates"
    )
    cross_episode_reasoning: str = Field(
        description="Detailed editorial notes explaining how dialogue continuity and dramatic contrast were preserved across the stitched segments"
    )

print("[OK] Strict Pydantic Schemas compiled successfully.")


[OK] Strict Pydantic Schemas compiled successfully.


## 2. Ingest AST & Extract Context Store
Loads the canonical AST (`01_structured_episodes.json`) and the enriched narrative extractions (`02_batch_extractions.json`).  
Executes a lightweight case-insensitive substring search over AST text and CONTINUITY NOTES for missing character/prop keywords.


In [4]:
# Load canonical AST and extractions
ast_path = WORKSPACE_DIR / "01_structured_episodes.json"
extractions_path = WORKSPACE_DIR / "02_batch_extractions.json"

assert ast_path.exists(), f"AST missing at {ast_path}"
assert extractions_path.exists(), f"Extractions missing at {extractions_path}"

ast_data = json.loads(ast_path.read_text(encoding="utf-8"))
extractions_data = json.loads(extractions_path.read_text(encoding="utf-8"))

# Build element lookup dictionary
element_lookup = {}
for ep in ast_data.get("episodes", []):
    for sc in ep.get("scenes", []):
        for el in sc.get("elements", []):
            element_lookup[el["element_id"]] = el

print(f"Total Episodes Loaded        : {len(ast_data.get('episodes', []))}")
print(f"Total AST Elements Indexed   : {len(element_lookup)}")
print(f"Narrative Moments Available  : {len(extractions_data.get('narrative_moments', []))}")
print(f"Single-Episode Candidates    : {len(extractions_data.get('candidates', []))}")

# Lightweight Substring Search Demo
def keyword_substring_search(query: str, ast_dict: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Case-insensitive substring search over AST text and CONTINUITY NOTES."""
    results = []
    q_lower = query.lower()
    for ep in ast_dict.get("episodes", []):
        ep_num = ep["episode_num"]
        notes = ep.get("header", {}).get("continuity_notes", "")
        notes_match = q_lower in notes.lower()
        for sc in ep.get("scenes", []):
            for el in sc.get("elements", []):
                content = el.get("content", "")
                if q_lower in content.lower() or notes_match:
                    results.append({
                        "episode": ep_num,
                        "element_id": el["element_id"],
                        "tag": el.get("raw_tag", el.get("element_type")),
                        "content": content[:80] + "..." if len(content) > 80 else content
                    })
    return results

sample_prop_hits = keyword_substring_search("signet", ast_data)
print(f"\nKeyword Substring Search ('signet') found {len(sample_prop_hits)} matches. Top 3:")
for h in sample_prop_hits[:3]:
    print(f"  Ep {h['episode']:02d} | {h['element_id']} | {h['tag']}: {h['content']}")


Total Episodes Loaded        : 50
Total AST Elements Indexed   : 3727
Narrative Moments Available  : 11
Single-Episode Candidates    : 10

Keyword Substring Search ('signet') found 101 matches. Top 3:
  Ep 12 | EP12_EL034 | CLARISSA (Theatrical, Vindictive): "My emerald signet ring fell deep into the frozen blackthorn briars at Ashworth ...
  Ep 13 | EP13_EL071 | CAMERA: Extreme close-up of her bloody hand closing around the emerald signet ring.
  Ep 14 | EP14_EL002 | ACTION: Juliana's blood-streaked, mud-caked hand slowly uncurls amidst the thorns, revea...


## 3. Improvement 1.1: Semantic Retrieval Using Embeddings
### The Limitation of Keyword Matching:
Screenplay events often express identical underlying narrative developments using completely different words:
- **Episode 4:** *"The soldier presents the ancient silver signet ring with the Ashburn crest..."*
- **Episode 42:** *"The Duke's ancestral seal is presented before the court..."*

Keyword search for `"signet"` fails to match Episode 42.  
**Solution:** Generate embeddings for narrative moments and calculate pairwise cosine similarity:
$$\operatorname{sim}(A, B) = \frac{A \cdot B}{\|A\| \|B\|} = \frac{\sum_{i=1}^n A_i B_i}{\sqrt{\sum_{i=1}^n A_i^2} \sqrt{\sum_{i=1}^n B_i^2}}$$

We retrieve the top 3–5 cross-episode pairs, look up their exact verbatim dialogue in the AST, and inject them into the synthesis prompt.


In [6]:
from phase4_synthesis.synthesizer import SemanticMomentRetriever

cache_file = PHASE4_DIR / "embeddings_cache.json"
retriever = SemanticMomentRetriever(
    cache_path=cache_file,
    api_key=os.environ.get("GEMINI_API_KEY")
)

moments = extractions_data.get("narrative_moments", [])
retriever.compute_all_moment_embeddings(moments)

# Compute cross-episode semantic affinities (min episode gap >= 3)
affinity_pairs = []
for m1 in moments:
    related = retriever.retrieve_top_related_moments(
        source_moment=m1,
        candidate_pool=moments,
        top_k=2,
        min_episode_gap=3
    )
    for m2, sim in related:
        key = tuple(sorted([m1["moment_id"], m2["moment_id"]]))
        affinity_pairs.append({
            "pair_key": key,
            "m1_id": m1["moment_id"],
            "m1_ep": m1["episode"],
            "m1_summary": m1["summary"][:65] + "...",
            "m2_id": m2["moment_id"],
            "m2_ep": m2["episode"],
            "m2_summary": m2["summary"][:65] + "...",
            "similarity": round(sim, 4),
            "moment_1": m1,
            "moment_2": m2
        })

# Deduplicate
unique_affinities = {}
for p in affinity_pairs:
    if p["pair_key"] not in unique_affinities:
        unique_affinities[p["pair_key"]] = p

df_affinities = pd.DataFrame(list(unique_affinities.values())).sort_values(by="similarity", ascending=False)
print(f"[SEMANTIC SEARCH] Discovered {len(df_affinities)} cross-episode affinities.")
df_affinities[["m1_id", "m1_ep", "m2_id", "m2_ep", "similarity", "m1_summary", "m2_summary"]].head(8)


[CACHE] Loaded 11 cached moment embeddings from embeddings_cache.json
[SEMANTIC SEARCH] Discovered 14 cross-episode affinities.


## 4. Verbatim AST Source Element Grounding & Prompt Builder
Before calling the LLM, Python retrieves the exact source screenplay elements (`source_elements_verbatim`) from `01_structured_episodes.json` and attaches them to the prompt so the LLM verifies actual dialogue continuity rather than relying on summaries.


In [8]:
from phase4_synthesis.synthesizer import CrossEpisodeSynthesizerEngine

engine = CrossEpisodeSynthesizerEngine(
    ast_path=ast_path,
    extractions_path=extractions_path,
    output_path=PHASE4_DIR / "03_synthesized_candidates.json",
    embeddings_cache_path=cache_file
)

top_semantic_pairs = list(unique_affinities.values())[:4]
sample_prompt = engine.build_synthesis_prompt(
    semantic_pairs=top_semantic_pairs,
    narrative_context=extractions_data.get("narrative_context", {})
)

print("=" * 75)
print(f"SYNTHESIS PROMPT PREVIEW ({len(sample_prompt.splitlines())} lines):")
print("=" * 75)
for line in sample_prompt.splitlines()[:28]:
    print(line)
print("... [Truncated for brevity] ...")


[CACHE] Loaded 11 cached moment embeddings from embeddings_cache.json
SYNTHESIS PROMPT PREVIEW (204 lines):
You are the Lead Promo Director and Video Producer for Kuku FM's high-stakes microdrama series.
Your task is to execute CROSS-EPISODE SYNTHESIS (composition_mode: "multi_episode") by stitching 2-3 segments across different episodes into high-converting promo candidates.

### THE THREE STANDARDIZED PROMO ARCHETYPES:
1. `immediate_shock_opener`: Segment 1 opens within <= 3.00s on a visceral physical action, slap, betrayal accusation, or explosive threat; Segment 2/3 cuts on an unresolved cliffhanger.
2. `visual_anomaly_mystery_escalation`: Segment 1 opens within <= 3.00s on a suspicious visual beat in CAMERA or ACTION (e.g. signet ring, royal crest, mysterious scar) that escalates to a confrontation twist across episodes.
3. `emotional_reversal_turning_point`: Segment 1 establishes a sharp status humiliation or vow (e.g. jilted at altar); Segment 2/3 delivers a dramatic power inver

## 5. Improvement 1.3: LLM Execution with Controlled Retries
If the LLM response is malformed, missing fields, or fails schema validation:
1. Capture the original response.
2. Identify the specific Pydantic schema validation errors.
3. Retry once with a targeted, concise correction prompt.
4. If it fails a second time, log and fall back to safe deterministic synthesis.
This prevents a single malformed response from breaking the pipeline.


In [10]:
# Execute synthesis with controlled retries
has_api_key = bool(os.environ.get("GEMINI_API_KEY"))

if has_api_key:
    print("[API CALL] Executing synthesis via Gemini with Controlled Retry harness...")
    synthesis_response = engine.execute_synthesis_llm_call(sample_prompt, retries=1)
else:
    print("[INFO] No GEMINI_API_KEY detected. Executing deterministic semantic synthesis engine...")
    synthesis_response = engine._deterministic_semantic_synthesis()

print(f"\nSynthesized Candidates Generated: {len(synthesis_response.candidates)}")
print(f"Editorial Reasoning Summary: {synthesis_response.cross_episode_reasoning[:150]}...")


[INFO] No GEMINI_API_KEY detected. Executing deterministic semantic synthesis engine...
[SYNTHESIS] Running deterministic semantic synthesis engine grounded in AST...

Synthesized Candidates Generated: 3
Editorial Reasoning Summary: Synthesized 3 distinct multi-episode candidates connecting baseline humiliation, visual prop identity, and violent confrontation across 50 episodes wi...


## 6. Strict Mechanical Invariant Verification Gate
Every synthesized candidate must mechanically pass:
1. **Element ID Invariant:** All `start_element_id` and `end_element_id` exist in the canonical AST.
2. **Hook Timing Invariant:** $0.0\text{s} \le \text{hook\_timestamp\_offset\_seconds} \le 3.0\text{s}$.
3. **Combined Duration Invariant:** $20.00\text{s} \le \text{total\_duration\_seconds} \le 60.00\text{s}$ ($2,000 \le \text{total\_duration\_cs} \le 6,000$).
4. **Multi-Episode Invariant:** Must stitch across $\ge 2$ distinct episodes.


In [12]:
verification_reports = []
verified_candidates = []

for cand in synthesis_response.candidates:
    is_valid, errors = engine.mechanically_verify_candidate(cand)
    episodes = [seg.episode_num for seg in cand.segments]
    verification_reports.append({
        "Candidate ID": cand.candidate_id,
        "Archetype": cand.archetype,
        "Relationship": cand.relationship_type,
        "Episodes": str(episodes),
        "Total Dur (s)": cand.total_duration_seconds,
        "Total Dur (cs)": cand.total_duration_cs,
        "Hook Offset": f"{cand.hook_timestamp_offset_seconds:.2f}s",
        "Valid": "PASSED" if is_valid else "FAILED",
        "Errors": "; ".join(errors) if errors else "None"
    })
    if is_valid:
        verified_candidates.append(cand)

df_verify = pd.DataFrame(verification_reports)
print("=" * 80)
print("MECHANICAL INVARIANT VERIFICATION AUDIT:")
print("=" * 80)
display(df_verify)

assert all(r["Valid"] == "PASSED" for r in verification_reports), "Discovered mechanical validation errors!"
print("\n[SUCCESS] 100% of synthesized candidates passed all mechanical invariants.")


MECHANICAL INVARIANT VERIFICATION AUDIT:
     Candidate ID                          Archetype Relationship Episodes  Total Dur (s)  Total Dur (cs) Hook Offset   Valid Errors
0  SYNTH_MULTI_01   emotional_reversal_turning_point     thematic  [1, 42]           40.7            4070       1.20s  PASSED   None
1  SYNTH_MULTI_02  visual_anomaly_mystery_escalation      implied  [4, 42]           31.7            3170       0.50s  PASSED   None
2  SYNTH_MULTI_03             immediate_shock_opener     explicit  [7, 41]           25.5            2550       1.00s  PASSED   None

[SUCCESS] 100% of synthesized candidates passed all mechanical invariants.


## 7. Improvement 1.4: Deep Dive into Precise Trims & Editing Instructions
Inspects segment-level actionable trim recommendations:
- Exact opening dialogue
- Exact ending dialogue
- Verbatim trim quotes
- Boundary types (`element_boundary` vs `internal_cut`)
- Editor waveform check flag
- Transitional stitching directives between episodes


In [14]:
sample_multi = verified_candidates[0]

print("=" * 75)
print(f"MULTI-EPISODE CANDIDATE: {sample_multi.candidate_id}")
print(f"Archetype         : {sample_multi.archetype}")
print(f"Relationship Type : {sample_multi.relationship_type}")
print(f"Total Duration    : {sample_multi.total_duration_seconds:.2f}s ({sample_multi.total_duration_cs} cs)")
print(f"Hook Offset       : {sample_multi.hook_timestamp_offset_seconds:.2f}s")
print(f"Overall Score     : {sample_multi.overall_score}/20")
print("=" * 75)

print("\n--- CONSTITUENT SEGMENTS & PRECISE TRIM RECOMMENDATIONS ---")
for seg in sample_multi.segments:
    trim = seg.trim_recommendation
    print(f"\n[SEGMENT {seg.segment_index}] Episode {seg.episode_num:02d} | [{seg.start_element_id} -> {seg.end_element_id}] | ({seg.duration_seconds:.2f}s)")
    print(f"  Role in Promo     : {seg.purpose_in_promo}")
    print(f"  Opening Dialogue  : {trim.exact_opening_dialogue}")
    print(f"  Ending Dialogue   : {trim.exact_ending_dialogue}")
    print(f"  Verbatim Cut Quote: {trim.verbatim_trim_quote}")
    print(f"  Boundary Type     : {trim.boundary_type}")
    print(f"  Needs Waveform Chk: {trim.requires_manual_editor_verification}")

print("\n--- TRANSITIONAL STITCHING INSTRUCTIONS (FOR VIDEO EDITORS) ---")
for idx, instruction in enumerate(sample_multi.transition_instructions, 1):
    print(f"  Step {idx}: {instruction}")


MULTI-EPISODE CANDIDATE: SYNTH_MULTI_01
Archetype         : emotional_reversal_turning_point
Relationship Type : thematic
Total Duration    : 40.70s (4070 cs)
Hook Offset       : 1.20s
Overall Score     : 20/20

--- CONSTITUENT SEGMENTS & PRECISE TRIM RECOMMENDATIONS ---

[SEGMENT 1] Episode 01 | [EP01_EL010 -> EP01_EL019] | (15.50s)
  Role in Promo     : Hook / Initial Humiliation at Altar
  Opening Dialogue  : I cannot wed a woman whose family name is ruined.
  Ending Dialogue   : Gerald drops Juliana's hand at the altar steps as the congregation gasps.
  Verbatim Cut Quote: I cannot wed a woman whose family name is ruined.
  Boundary Type     : element_boundary
  Needs Waveform Chk: False

[SEGMENT 2] Episode 42 | [EP42_EL001 -> EP42_EL014] | (25.20s)
  Role in Promo     : Climax / Payoff & Full Ducal Vindication
  Opening Dialogue  : Alexander strides deliberately through the pulverized cathedral doors in full ducal armor.
  Ending Dialogue   : I am Alexander, Duke of Ashburn, and 

## 8. Improvement 1.2: Diversity-Aware Final Selection & Greedy Ranking
### Why Diversity Matters:
Sorting candidates solely by raw rubric scores often creates severe episode concentration (e.g. 10 recommendations from Episode 42).  
Our **Diversity-Aware Greedy Selection Algorithm** balances quality with representation:

| Constraint | Rule | Purpose |
| :--- | :--- | :--- |
| **Episode Concentration** | Max 3 recommendations per episode | Prevents marketing promo fatigue from one storyline |
| **Archetype Diversity** | Guaranteed representation of all 3 archetypes | Surfaces shock, visual mystery, and emotional reversal |
| **Composition Diversity** | Includes both single-episode & multi-episode clips | Showcases both tight scenes and epic cross-series arcs |
| **Quality Priority** | Never selects below quality threshold (score $\ge 14$) | Retains high conversion as the primary standard |


In [16]:
from phase4_synthesis.synthesizer import diversity_aware_greedy_selection

single_cands = extractions_data.get("candidates", [])
multi_cands = [c.model_dump() for c in verified_candidates]

diversity_results = diversity_aware_greedy_selection(
    single_candidates=single_cands,
    multi_candidates=multi_cands,
    target_count=15,
    max_per_episode=3,
    min_quality_score=14
)

print("=" * 75)
print("DIVERSITY-AWARE SELECTION AUDIT:")
print("=" * 75)
print(f"Target Selection Count   : {diversity_results['target_count']}")
print(f"Actual Candidates Chosen : {diversity_results['selected_count']}")
print(f"Archetype Distribution   : {diversity_results['archetype_distribution']}")
print(f"Composition Distribution : {diversity_results['composition_distribution']}")
print(f"Episode Counts Breakdown : {diversity_results['episode_distribution']}")
print("=" * 75)

# Format Top 15 Selection Table
top_15_rows = []
for idx, c in enumerate(diversity_results["selected_candidates"], 1):
    top_15_rows.append({
        "Rank": idx,
        "Candidate ID": c.get("candidate_id", c.get("start_element_id")),
        "Mode": c.get("composition_mode"),
        "Archetype": c.get("archetype"),
        "Episodes": str(c.get("episodes_involved")),
        "Score": c.get("overall_score"),
        "Opening Dialogue": c["trim_recommendation"]["exact_opening_dialogue"][:45] + "..." if "trim_recommendation" in c else c["segments"][0]["trim_recommendation"]["exact_opening_dialogue"][:45] + "..."
    })

df_top15 = pd.DataFrame(top_15_rows)
display(df_top15)


DIVERSITY-AWARE SELECTION AUDIT:
Target Selection Count   : 15
Actual Candidates Chosen : 13
Archetype Distribution   : {'immediate_shock_opener': 5, 'visual_anomaly_mystery_escalation': 3, 'emotional_reversal_turning_point': 5}
Composition Distribution : {'single_episode': 10, 'multi_episode': 3}
Episode Counts Breakdown : {1: 2, 3: 1, 42: 3, 7: 2, 41: 2, 25: 1, 4: 2, 2: 1, 50: 1, 12: 1}
    Rank    Candidate ID            Mode                          Archetype Episodes  Score                                  Opening Dialogue
0      1      EP01_EL010  single_episode             immediate_shock_opener      [1]     20  I cannot wed a woman whose family name is rui...
1      2      EP03_EL055  single_episode  visual_anomaly_mystery_escalation      [3]     20  Stand down before these stones drink your blo...
2      3      EP42_EL001  single_episode   emotional_reversal_turning_point     [42]     20     Who are you to interrupt a ducal ceremony?...
3      4  SYNTH_MULTI_01   multi_episode

## 9. Deliverables Export & Mechanical Verification Summary
Emits `03_synthesized_candidates.json` containing:
- All verified multi-episode synthesized candidates
- Semantic retrieval audit log (cosine similarities)
- Mechanical validation audit log (100% invariants checked)
- Diversity-aware Top 15 recommendations ready for Phase 5 (`validator_ranker.py`)


In [18]:
# Compile final payload
final_payload = {
    "series_title": extractions_data.get("series_title", "Microdrama Series"),
    "total_synthesized_candidates": len(verified_candidates),
    "synthesis_notes": synthesis_response.cross_episode_reasoning,
    "synthesized_candidates": [c.model_dump() for c in verified_candidates],
    "diversity_ranked_top15": diversity_results["selected_candidates"],
    "diversity_metadata": {
        "archetype_distribution": diversity_results["archetype_distribution"],
        "composition_distribution": diversity_results["composition_distribution"],
        "episode_distribution": diversity_results["episode_distribution"]
    },
    "semantic_retrieval_audit": list(unique_affinities.values())[:10],
    "mechanical_audit": verification_reports
}

# Export to phase4_synthesis and root
out_phase4 = PHASE4_DIR / "03_synthesized_candidates.json"
out_root = WORKSPACE_DIR / "03_synthesized_candidates.json"

out_phase4.write_text(json.dumps(final_payload, indent=2), encoding="utf-8")
out_root.write_text(json.dumps(final_payload, indent=2), encoding="utf-8")

print("=" * 75)
print("PHASE 4: CROSS-EPISODE SYNTHESIS COMPLETE")
print("=" * 75)
print(f"Exported Phase 4 Artifact: {out_phase4.resolve()} ({out_phase4.stat().st_size / 1024:.1f} KB)")
print(f"Mirrored Root Artifact   : {out_root.resolve()}")
print(f"Synthesized Candidates   : {len(verified_candidates)}")
print(f"Diversity Top 15 Pool    : {len(diversity_results['selected_candidates'])}")
print("=" * 75)


PHASE 4: CROSS-EPISODE SYNTHESIS COMPLETE
Exported Phase 4 Artifact: C:\Users\Anirudh Singh\OneDrive\Documents\Kuku FM\phase4_synthesis\03_synthesized_candidates.json (80.0 KB)
Mirrored Root Artifact   : C:\Users\Anirudh Singh\OneDrive\Documents\Kuku FM\03_synthesized_candidates.json
Synthesized Candidates   : 3
Diversity Top 15 Pool    : 13
